# 谷子 1844 种质 · 基因×样本单倍型频率整合矩阵

对标 `coo-1-0-1-4-Rice_18k-get-hap-freq.ipynb`：把 `2-allGeneHapPer/` 中
每基因一个文件整合成一个 **样本 × 基因** 的 percentage 矩阵。

**与参考脚本的关键差异（不可照搬处）**：
- 参考取基因名用 `x.split(".")[0]`——Rice 文件名（`LOC_Os01g10110.txt`）中点号只在 `.txt`。
  谷子有 75 个文件名本身含点（`AAP6__Seita.2G026700.txt`），`split(".")[0]` 会截断成
  `AAP6__Seita`，4 个 AAP6 文件互相覆盖成同名列 → **必须用 `pathlib.Path.stem`**（只去 `.txt`）
- `pd.read_table` 在 pandas 2.x 已弃用 → `pd.read_csv(sep="\t")`
- 列顺序按 gene_list.txt（染色体号 → Start），而非参考的 `os.listdir` 任意序

**输出**：`Millet_1k_hap_per.txt`（1844 行 × 306 列，Tab，首列表头 `sample`）

In [1]:
import pathlib

import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd()
HAPPER_PATH = ROOT / "2-allGeneHapPer"
GENE_LIST_FILE = ROOT / "gene_list.txt"
OUT_FILE = ROOT / "Millet_1k_hap_per.txt"

In [2]:
# 列顺序 = gene_list.txt 行序（染色体号 → Start），基因名 = File 列去 .xls
gl = pd.read_csv(GENE_LIST_FILE, sep="\t", dtype=str)
gene_order = [f[:-4] for f in gl["File"]]

# 一致性断言：2-allGeneHapPer 的文件集合必须与 gene_list 完全一致
actual = sorted(p.stem for p in HAPPER_PATH.glob("*.txt"))  # stem 只去 .txt，保留内部点号
assert actual == sorted(gene_order), \
    f"文件集合不一致: 仅在HapPer={set(actual)-set(gene_order)}, 仅在gene_list={set(gene_order)-set(actual)}"
assert len(gene_order) == len(set(gene_order)) == 306, "基因名存在重复"

# 逐文件读 percentage 列（index=sample），列名改为基因名
df_dict = {}
for item in gene_order:
    tmp = pd.read_csv(HAPPER_PATH / f"{item}.txt", sep="\t", index_col="sample")[["percentage"]]
    tmp.columns = [item]
    df_dict[item] = tmp

# 一次性按列拼接（dict 保持插入顺序 = gene_order）
df_per = pd.concat(df_dict.values(), axis=1)
df_per.index.name = "sample"

df_per.to_csv(OUT_FILE, sep="\t")
print(f"形状: {df_per.shape}")
print(f"已写入: {OUT_FILE.name}  ({OUT_FILE.stat().st_size:,} bytes)")
df_per.iloc[:4, :5]

形状: (1844, 306)
已写入: Millet_1k_hap_per.txt  (10,682,414 bytes)


,NRPC2,GW2__Seita.1G020400,FT__Seita.1G026000,MED18,AGL61
sample,,,,,
WGS0083,0.030911,0.027657,0.65564,0.045011,0.633948
WGS0320,0.330803,0.406725,0.65564,0.602495,0.633948
WGS0323,0.330803,0.106291,0.65564,0.602495,0.633948
WGS0325,0.330803,0.106291,0.65564,0.602495,0.633948


In [3]:
# ==== 验证 ====
checks = [
    ("形状 = (1844, 306)", df_per.shape == (1844, 306)),
    ("无空值", bool(df_per.notna().all().all())),
    ("值域 (0, 1]", bool(((df_per > 0) & (df_per <= 1)).all().all())),
    ("306 列名互不重复且按 gene_list 顺序", df_per.columns.is_unique and list(df_per.columns) == gene_order),
    ("1844 行索引唯一", df_per.index.is_unique),
]

# 全量逐列比对：每列与源 HapPer 文件的 percentage 逐样本全等（非抽查）
bad = []
for item in gene_order:
    src = pd.read_csv(HAPPER_PATH / f"{item}.txt", sep="\t", index_col="sample")["percentage"]
    if not np.array_equal(df_per[item].to_numpy(), src.to_numpy()):
        bad.append(item)
checks.append(("全部 306 列与源 HapPer 逐样本全等", not bad))
if bad:
    print("  不一致列:", bad[:5])

# 回读落盘文件，确认写盘无损
back = pd.read_csv(OUT_FILE, sep="\t", index_col="sample")
checks.append(("落盘文件回读与内存完全一致", back.shape == (1844, 306) and np.array_equal(back.to_numpy(), df_per.to_numpy())))

print("=== 验收 ===")
allpass = True
for desc, okv in checks:
    allpass &= bool(okv)
    print(f"  [{'PASS' if okv else 'FAIL'}] {desc}")
print()
print("验收结论:", "全部通过" if allpass else "存在失败项")

=== 验收 ===
  [PASS] 形状 = (1844, 306)
  [PASS] 无空值
  [PASS] 值域 (0, 1]
  [PASS] 306 列名互不重复且按 gene_list 顺序
  [PASS] 1844 行索引唯一
  [PASS] 全部 306 列与源 HapPer 逐样本全等
  [PASS] 落盘文件回读与内存完全一致

验收结论: 全部通过
